# 🚗 TransFusion-L 3D Object Detection trên Google Colab (Bản Chuẩn Hoá Toàn Diện v2)

Notebook này đã được cấu hình lại toàn bộ pipeline để triệt tiêu các lỗi:
1. **Python 3.13 incompatibility**: Dùng môi trường Python 3.10 thông qua `micromamba` để `spconv` và `cumm` dùng binary `.so` dựng sẵn.
2. **Lỗi `setuptools` mới làm gãy `setup.py develop`**: Hạ cấp `setuptools<69` và dùng cờ `--no-build-isolation` khi cài OpenPCDet.
3. **Parser PCD**: Hỗ trợ an toàn cả định dạng PCD Binary và ASCII.

**Yêu cầu:** Chọn menu **Runtime** → **Change runtime type** → **T4 GPU**.

## Bước 0: Kiểm tra GPU phần cứng

In [1]:
!nvidia-smi

Mon Oct  5 13:28:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bước 1: Khởi tạo môi trường Python 3.10 & Thư viện (spconv, PyTorch, setuptools tương thích)

In [2]:
# 1. Tải và giải nén micromamba
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba

# 2. Khởi tạo môi trường Python 3.10
!./bin/micromamba create -y -p /content/env python=3.10 -c conda-forge

# 3. Khóa bản setuptools & wheel ổn định để build C++/CUDA extension của OpenPCDet không bị lỗi PEP 517
!./bin/micromamba run -p /content/env pip install "setuptools<69.0.0" wheel ninja

# 4. Cài PyTorch cu121 & spconv dựng sẵn
!./bin/micromamba run -p /content/env pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!./bin/micromamba run -p /content/env pip install spconv-cu120 gdown pyyaml numba scipy opencv-python

# 5. Kiểm tra môi trường
!./bin/micromamba run -p /content/env python -c "import torch, spconv; print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| spconv:', spconv.__version__)"

[+] 0.0s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                     ✔ Done (6.8 sec)

Resolving Environment                                                                     ✔ Done (0.3 sec)

Transaction

  Prefix: /content/env

  Updating specs:

   - python=3.10
   - pip


  Package               Version  Build                 Channel          Size
──────────────────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge      29kB
  + bzip2                 1.0.8  hda65f42_10           conda-forge     258kB
  + ca-certificates   2026.7.22  hbd8a1cb_0            conda-forge     132kB

## Bước 2: Clone & Biên dịch OpenPCDet

In [8]:
import os

if not os.path.exists('/content/OpenPCDet'):
    !git clone https://github.com/open-mmlab/OpenPCDet.git /content/OpenPCDet

# 1. Cài các dependency trước
!./bin/micromamba run -p /content/env pip install easydict SharedArray tensorboardX scikit-image

# 2. Xóa pyproject.toml tránh lỗi PEP 517 build
!rm -f /content/OpenPCDet/pyproject.toml

# 3. Patch đè hàm kiểm tra CUDA của PyTorch bằng Python thuần (không dùng regex)
patch_code = """
import torch.utils.cpp_extension
torch.utils.cpp_extension._check_cuda_version = lambda *args, **kwargs: None
"""
# Ghi patch thẳng vào đầu file setup.py của OpenPCDet
setup_file = "/content/OpenPCDet/setup.py"
with open(setup_file, "r") as f:
    orig_content = f.read()

with open(setup_file, "w") as f:
    f.write(patch_code + "\n" + orig_content)

print("✓ Đã chèn patch bỏ qua check CUDA trực tiếp vào OpenPCDet/setup.py")

# 4. Biên dịch CUDA extensions cho Tesla T4 (sm_75)
!cd /content/OpenPCDet && \
  TORCH_CUDA_ARCH_LIST="7.5" \
  FORCE_CUDA="1" \
  /content/bin/micromamba run -p /content/env python setup.py build_ext --inplace

# 5. Cài đặt OpenPCDet vào môi trường
!cd /content/OpenPCDet && \
  /content/bin/micromamba run -p /content/env python setup.py develop --no-deps

✓ Đã chèn patch bỏ qua check CUDA trực tiếp vào OpenPCDet/setup.py
running build_ext
building 'pcdet.ops.iou3d_nms.iou3d_nms_cuda' extension
creating /content/OpenPCDet/build
creating /content/OpenPCDet/build/temp.linux-x86_64-cpython-310
creating /content/OpenPCDet/build/temp.linux-x86_64-cpython-310/pcdet
creating /content/OpenPCDet/build/temp.linux-x86_64-cpython-310/pcdet/ops
creating /content/OpenPCDet/build/temp.linux-x86_64-cpython-310/pcdet/ops/iou3d_nms
creating /content/OpenPCDet/build/temp.linux-x86_64-cpython-310/pcdet/ops/iou3d_nms/src
Emitting ninja build file /content/OpenPCDet/build/temp.linux-x86_64-cpython-310/build.ninja...
Compiling objects...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
[1/4] c++ -MMD -MF /content/OpenPCDet/build/temp.linux-x86_64-cpython-310/pcdet/ops/iou3d_nms/src/iou3d_cpu.o.d -pthread -B /content/env/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall 

## Bước 3: Tải Pretrained Weights TransFusion-L

In [9]:
!mkdir -p /content/checkpoints

import os
ckpt_path = '/content/checkpoints/transfusion_lidar.pth'
if not os.path.exists(ckpt_path):
    !./bin/micromamba run -p /content/env gdown "1cuZ2qdDnxSwTCsiXWwbqCGF-uoazTXbz" -O {ckpt_path}
else:
    print("Checkpoint đã tồn tại, bỏ qua tải.")

!ls -lh /content/checkpoints/

Downloading...
From (original): https://drive.google.com/uc?id=1cuZ2qdDnxSwTCsiXWwbqCGF-uoazTXbz
From (redirected): https://drive.google.com/uc?id=1cuZ2qdDnxSwTCsiXWwbqCGF-uoazTXbz&confirm=t&uuid=1ce0c954-0013-49bc-8d84-fbf398b49c0d
To: /content/checkpoints/transfusion_lidar.pth

total 32M
-rw------- 1 root root 32M May  9  2023 transfusion_lidar.pth


## Bước 4: Upload dữ liệu PCD

Upload các file `.pcd` vào thư mục `/content/data/`.

In [10]:
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
print("Chọn tất cả file .pcd để upload (hoặc kéo thả vào panel Files bên trái)...\n")
uploaded = files.upload()

for filename in uploaded:
    os.rename(filename, f'/content/data/{filename}')
    print(f"  ✓ {filename}")

print(f"\nTổng cộng: {len(os.listdir('/content/data'))} files trong /content/data/")

Chọn tất cả file .pcd để upload (hoặc kéo thả vào panel Files bên trái)...



Saving 20.pcd to 20.pcd
Saving 21.pcd to 21.pcd
Saving 22.pcd to 22.pcd
Saving 23.pcd to 23.pcd
Saving 24.pcd to 24.pcd
Saving 25.pcd to 25.pcd
Saving 26.pcd to 26.pcd
Saving 27.pcd to 27.pcd
Saving 28.pcd to 28.pcd
Saving 29.pcd to 29.pcd
  ✓ 20.pcd
  ✓ 21.pcd
  ✓ 22.pcd
  ✓ 23.pcd
  ✓ 24.pcd
  ✓ 25.pcd
  ✓ 26.pcd
  ✓ 27.pcd
  ✓ 28.pcd
  ✓ 29.pcd

Tổng cộng: 10 files trong /content/data/


## Bước 5: Viết script Inference & Heading Correction

In [21]:
%%writefile /content/run_transfusion.py
import os
import sys
import glob
import json
import math
import numpy as np
from pathlib import Path
from scipy.optimize import linear_sum_assignment

# Patch tương thích kiểu dữ liệu cũ
for attr in ['int', 'float', 'long']:
    if not hasattr(np, attr):
        setattr(np, attr, int if attr != 'float' else float)
if not hasattr(np, 'bool'):
    np.bool = np.bool_

import torch
sys.path.insert(0, '/content/OpenPCDet')
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets.dataset import DatasetTemplate
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

def read_pcd_points(pcd_path):
    with open(pcd_path, 'rb') as f:
        data_type = 'ascii'
        while True:
            line = f.readline().decode('ascii', errors='ignore')
            if line.startswith('DATA'):
                if 'binary' in line.lower():
                    data_type = 'binary'
                break
        if data_type == 'binary':
            points = np.fromfile(f, dtype=np.float32).reshape(-1, 4)
        else:
            f.seek(0)
            lines = [l.strip() for l in f.readlines() if l.strip() and not l.startswith(b'#')]
            data_start = 0
            for idx, l in enumerate(lines):
                if l.startswith(b'DATA'):
                    data_start = idx + 1
                    break
            points = np.loadtxt(lines[data_start:], dtype=np.float32)[:, :4]
    return points

class PCDSequenceDataset(DatasetTemplate):
    def __init__(self, dataset_cfg, class_names, pcd_dir):
        super().__init__(dataset_cfg=dataset_cfg, class_names=class_names, training=False, root_path=Path('.'), logger=None)
        def extract_key(f):
            base = os.path.splitext(os.path.basename(f))[0]
            return int(base) if base.isdigit() else base
        self.pcd_files = sorted(glob.glob(os.path.join(pcd_dir, '*.pcd')), key=extract_key)

    def __len__(self):
        return len(self.pcd_files)

    def __getitem__(self, index):
        pcd_path = self.pcd_files[index]
        base_name = os.path.splitext(os.path.basename(pcd_path))[0]
        frame_id = int(base_name) if base_name.isdigit() else index

        points = read_pcd_points(pcd_path)
        timestamps = np.zeros((points.shape[0], 1), dtype=np.float32)
        points = np.hstack([points, timestamps])

        input_dict = {'points': points, 'frame_id': frame_id}
        return self.prepare_data(data_dict=input_dict)

VEHICLE_CLASSES = {'car', 'truck', 'bus', 'trailer', 'construction_vehicle', 'motorcycle', 'bicycle'}

def _angle_diff(a, b):
    d = a - b
    return (d + math.pi) % (2 * math.pi) - math.pi

def correct_heading(all_results, max_match_dist=4.0, vel_min_speed=3.0, vel_max_cos=-0.7, vel_min_matches=3, con_radius=10.0):
    sorted_results = sorted(all_results, key=lambda r: r['frame'])
    n = len(sorted_results)
    if n < 2:
        return all_results

    # PHASE 1: Velocity Hungarian Matching
    velocity_evidence = {}
    for fi in range(n - 1):
        curr_boxes = sorted_results[fi]['boxes']
        next_boxes = sorted_results[fi + 1]['boxes']
        if not curr_boxes or not next_boxes:
            continue
        curr_xy = np.array([b['center'][:2] for b in curr_boxes])
        next_xy = np.array([b['center'][:2] for b in next_boxes])

        cost = np.full((len(curr_boxes), len(next_boxes)), 1e6)
        for ci in range(len(curr_boxes)):
            for ni in range(len(next_boxes)):
                if curr_boxes[ci]['label'] == next_boxes[ni]['label']:
                    cost[ci, ni] = np.linalg.norm(curr_xy[ci] - next_xy[ni])

        row_ind, col_ind = linear_sum_assignment(cost)
        displacements, valid_matches = [], []
        for ri, ci in zip(row_ind, col_ind):
            if cost[ri, ci] > max_match_dist:
                continue
            disp = next_xy[ci] - curr_xy[ri]
            displacements.append(disp)
            valid_matches.append((ri, ci, disp))

        if len(displacements) < vel_min_matches:
            continue

        ego_disp = np.median(np.vstack(displacements), axis=0)
        for ri, ci, disp in valid_matches:
            true_vel = disp - ego_disp
            velocity_evidence.setdefault((fi, ri), []).append(true_vel)
            velocity_evidence.setdefault((fi + 1, ci), []).append(true_vel)

    vel_flips = 0
    for (fi, bi), vels in velocity_evidence.items():
        avg_vel = np.mean(np.vstack(vels), axis=0)
        speed = np.linalg.norm(avg_vel)
        if speed < vel_min_speed:
            continue
        box = sorted_results[fi]['boxes'][bi]
        if box['label'] not in VEHICLE_CLASSES:
            continue
        yaw = box['yaw']
        heading_vec = np.array([np.cos(yaw), np.sin(yaw)])
        cos_angle = np.dot(heading_vec, avg_vel / speed)
        if cos_angle < vel_max_cos:
            new_yaw = (yaw + math.pi + math.pi) % (2 * math.pi) - math.pi
            box['yaw'] = round(float(new_yaw), 3)
            box['heading_corrected'] = True
            vel_flips += 1

    # PHASE 2: Intra-frame Consistency
    cons_flips = 0
    for fi, frame_data in enumerate(sorted_results):
        boxes = frame_data['boxes']
        for bi, box in enumerate(boxes):
            if box['label'] not in VEHICLE_CLASSES or box.get('heading_corrected'):
                continue
            neighbors = [o for j, o in enumerate(boxes) if bi != j and o['label'] == box['label'] and np.linalg.norm(np.array(box['center'][:2]) - np.array(o['center'][:2])) < con_radius]
            if len(neighbors) < 2:
                continue
            my_yaw = box['yaw']
            opposed = sum(1 for nb in neighbors if abs(_angle_diff(my_yaw, nb['yaw'])) > 2.5)
            aligned = sum(1 for nb in neighbors if abs(_angle_diff(my_yaw, nb['yaw'])) < 0.7)
            if opposed >= 2 and aligned == 0:
                new_yaw = (my_yaw + math.pi + math.pi) % (2 * math.pi) - math.pi
                box['yaw'] = round(float(new_yaw), 3)
                box['heading_corrected'] = True
                cons_flips += 1

    print(f"Heading Correction: {vel_flips} flips (Velocity), {cons_flips} flips (Consistency)")
    return all_results

# Inference Setup
CFG_FILE = '/content/OpenPCDet/tools/cfgs/nuscenes_models/transfusion_lidar.yaml'
CKPT_FILE = '/content/checkpoints/transfusion_lidar.pth'
PCD_DIR = '/content/data'
OUTPUT_JSON = '/content/transfusion_predictions.json'
SCORE_THRESH = 0.20

cfg_from_yaml_file(CFG_FILE, cfg)
class_names = cfg.CLASS_NAMES
dataset = PCDSequenceDataset(dataset_cfg=cfg.DATA_CONFIG, class_names=class_names, pcd_dir=PCD_DIR)
logger = common_utils.create_logger()
model = build_network(model_cfg=cfg.MODEL, num_class=len(class_names), dataset=dataset)
model.load_params_from_file(filename=CKPT_FILE, logger=logger, to_cpu=False)
model.cuda()
model.eval()

all_results = []
print(f"Dang chay inference tren {len(dataset)} frames...")

with torch.no_grad():
    for i in range(len(dataset)):
        data_dict = dataset[i]
        frame_id = data_dict['frame_id']
        batch_dict = dataset.collate_batch([data_dict])
        load_data_to_gpu(batch_dict)
        pred_dicts, _ = model.forward(batch_dict)
        boxes = pred_dicts[0]['pred_boxes'].cpu().numpy()
        scores = pred_dicts[0]['pred_scores'].cpu().numpy()
        labels = pred_dicts[0]['pred_labels'].cpu().numpy()

        frame_boxes = []
        for b, s, l in zip(boxes, scores, labels):
            if float(s) < SCORE_THRESH:
                continue
            frame_boxes.append({
                'label': class_names[l - 1],
                'score': round(float(s), 3),
                'center': [round(float(b[0]), 3), round(float(b[1]), 3), round(float(b[2]), 3)],
                'size': [round(float(b[3]), 3), round(float(b[4]), 3), round(float(b[5]), 3)],
                'yaw': round(float(b[6]), 3)
            })
        all_results.append({'frame': frame_id, 'num_boxes': len(frame_boxes), 'boxes': frame_boxes})
        print(f"  Frame {frame_id}: {len(frame_boxes)} boxes")

print("Bat dau Heading Correction...")
all_results = correct_heading(all_results)

with open(OUTPUT_JSON, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\n✓ Thanh cong! Ket qua luu tai: {OUTPUT_JSON}")

Overwriting /content/run_transfusion.py


In [22]:
!./bin/micromamba run -p /content/env pip install pyquaternion nuscenes-devkit

In [23]:
!sed -i 's/.*argo2_dataset.*/Argo2Dataset = None/' /content/OpenPCDet/pcdet/datasets/__init__.py

## Bước 6: Chạy Inference toàn bộ pipeline

In [24]:
!cd /content/OpenPCDet/tools && /content/bin/micromamba run -p /content/env python /content/run_transfusion.py

/content/run_transfusion.py:12: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, attr):
/content/run_transfusion.py:14: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'bool'):
/content/env/lib/python3.10/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/content/env/lib/python3.10/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/content/env/lib/python3.10/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_

## Bước 7: Xem thống kê & Tải file JSON về máy

In [25]:
import json
from collections import Counter
from google.colab import files

with open('/content/transfusion_predictions.json') as f:
    results = json.load(f)

total_boxes = sum(r['num_boxes'] for r in results)
print(f"=== TỔNG KẾT: {total_boxes} boxes trên {len(results)} frames ===\n")
for r in results:
    counts = Counter([b['label'] for b in r['boxes']])
    print(f"Frame {r['frame']}: {r['num_boxes']} boxes -> {dict(counts)}")

# Tải file JSON về máy
files.download('/content/transfusion_predictions.json')

=== TỔNG KẾT: 173 boxes trên 10 frames ===

Frame 20: 23 boxes -> {'car': 11, 'pedestrian': 9, 'motorcycle': 1, 'bus': 1, 'truck': 1}
Frame 21: 15 boxes -> {'car': 11, 'motorcycle': 1, 'pedestrian': 3}
Frame 22: 17 boxes -> {'car': 11, 'motorcycle': 1, 'pedestrian': 4, 'truck': 1}
Frame 23: 15 boxes -> {'motorcycle': 1, 'car': 11, 'truck': 1, 'pedestrian': 2}
Frame 24: 13 boxes -> {'car': 10, 'motorcycle': 1, 'pedestrian': 1, 'truck': 1}
Frame 25: 21 boxes -> {'car': 13, 'pedestrian': 6, 'truck': 1, 'motorcycle': 1}
Frame 26: 22 boxes -> {'car': 16, 'pedestrian': 4, 'motorcycle': 1, 'truck': 1}
Frame 27: 16 boxes -> {'pedestrian': 4, 'car': 11, 'truck': 1}
Frame 28: 16 boxes -> {'pedestrian': 4, 'car': 10, 'trailer': 1, 'truck': 1}
Frame 29: 15 boxes -> {'pedestrian': 6, 'car': 9}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>